In [1]:
# GPT-4o-based forcasting extration machine

In [ ]:
from openai import OpenAI
import os

def extract_json_with_openai(log_path, output_path):
    # 1. Read the API key from the environment
    #    Set it in your shell or a .env file first: export OPENAI_API_KEY="your-key"
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))

    # 2. Read the log contents
    with open(log_path, "r", encoding="utf-8") as f:
        log_content = f.read()

    # 3. Build the ChatCompletion request
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4o-mini" / "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You are an AI assistant skilled at processing text."},
            {
                "role": "user",
                "content": (
                    "Extract the first valid JSON object from the log below and return only that JSON.\n"
                    "Log contents:\n"
                    "-----\n"
                    f"{log_content}\n"
                    "-----\n"
                    "Return only the JSON object, with no explanation."
                ),
            },
        ],
        temperature=0,
        max_tokens=2048,
    )  # :contentReference[oaicite:0]{index=0}

    # 4. Take the model reply and write it to a file
    json_content = response.choices[0].message.content.strip()
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(json_content)

    print(f"Extracted JSON saved to {output_path}")


extract_json_with_openai(
    "D:/my-fin-project/outputs/full_run.log",
    "D:/my-fin-project/outputs/forcasting_result.txt"
 )

In [ ]:
# English output, network access as needed
result = generate_structured_outlook_notebook(
    topic="Nvidia",
    output_path="forcasting_result.json",
    context_source=None,   # or "notes.txt" / "https://example.com/notes.txt"
    model="gpt-5",         # fall back to "gpt-4o" if unavailable
    force_web=False,       # True = always force a web lookup
    max_news=10,
    lang="en",
    max_output_tokens=2048
)
result  # display the result inline


In [3]:
# Snapshot of pricing data of a stock

In [ ]:
import os
import requests
from datetime import datetime, timedelta

POLYGON_KEY = os.getenv("POLYGON_API_KEY", "")
BASE = "https://api.polygon.io"

def fetch_polygon_data(ticker: str) -> dict:
    # -- 1. Snapshot endpoint: everything in one call -- #
    snap_url = f"{BASE}/v2/snapshot/locale/us/markets/stocks/tickers/{ticker}"
    resp = requests.get(snap_url, params={"apiKey": POLYGON_KEY})
    resp.raise_for_status()
    snap = resp.json().get("ticker", {})

    # Uncomment the next line to inspect which fields snap contains:
    # print("Snapshot keys:", snap.keys())

    # Parse snapshot fields defensively
    last_price = snap.get("lastTrade", {}).get("p") \
                 or snap.get("day", {}).get("c")
    prev_close = snap.get("prevDay", {}).get("c")
    o          = snap.get("day", {}).get("o")
    h          = snap.get("day", {}).get("h")
    l          = snap.get("day", {}).get("l")
    v          = snap.get("day", {}).get("v")
    after_vol  = snap.get("afterHours", {}).get("v")
    bid_qty    = snap.get("lastQuote", {}).get("bq")
    bid_price  = snap.get("lastQuote", {}).get("bp")
    ask_qty    = snap.get("lastQuote", {}).get("aq")
    ask_price  = snap.get("lastQuote", {}).get("ap")
    bid         = f"{bid_price}×{bid_qty}" if bid_price and bid_qty else None
    ask         = f"{ask_price}×{ask_qty}" if ask_price and ask_qty else None

    # -- 2. Historical aggregates over 252 days: 52-week high/low and average volume -- #
    end_ms   = int(datetime.utcnow().timestamp() * 1000)
    start_ms = int((datetime.utcnow() - timedelta(days=365)).timestamp() * 1000)
    hist_url = (
        f"{BASE}/v2/aggs/ticker/{ticker}/range/1/day/"
        f"{start_ms}/{end_ms}"
    )
    resp2 = requests.get(hist_url, params={
        "adjusted": "true", "sort": "desc", "limit": 252, "apiKey": POLYGON_KEY
    })
    resp2.raise_for_status()
    hist = resp2.json().get("results", [])

    highs = [d["h"] for d in hist]
    lows  = [d["l"] for d in hist]
    vols  = [d["v"] for d in hist]

    return {
        "last_price":         last_price,
        "prev_close":         prev_close,
        "open":               o,
        "high":               h,
        "low":                l,
        "volume":             v,
        "after_hours_volume": after_vol,
        "avg_volume_252d":    sum(vols) / len(vols) if vols else None,
        "52w_high":           max(highs) if highs else None,
        "52w_low":            min(lows) if lows else None,
    }

if __name__ == "__main__":
    data = fetch_polygon_data("NVDA")
    for k, v in data.items():
        print(f"{k}: {v}")


In [5]:
# Plot Creation

In [6]:
import mplfinance as mpf
import pandas as pd
import requests
from datetime import datetime

API_KEY = os.getenv("POLYGON_API_KEY", "")
BASE_URL = "https://api.polygon.io"

stock = "NVDA"
timespan = "day"
start_date = "2025-03-03"
end_date = "2025-07-22"

def fetch_ohlcv(ticker, start_date, end_date):
    """
    Returns a DataFrame with DatetimeIndex and columns ['Open','High','Low','Close','Volume']
    """
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/{timespan}/{start_date}/{end_date}"
    params = {
        "adjusted": "true",
        "sort":     "asc",
        "limit":    5000,
        "apiKey":   API_KEY
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    bars = r.json().get("results", [])
    df = pd.DataFrame(bars)
    df['date'] = pd.to_datetime(df['t'], unit='ms')
    df.set_index('date', inplace=True)
    df = df[['o','h','l','c','v']]
    df.columns = ['Open','High','Low','Close','Volume']
    return df

df = fetch_ohlcv(stock, start_date, end_date)

mpf.plot(
    df,
    type='candle',
    volume=True,
    mav=(10, 20),
    title=f'{stock} Daily Candlestick Chart\n{start_date} to {end_date}',
    style='yahoo',
    tight_layout=True,
    savefig=f'outputs/chart/{stock.lower()}_candlestick.png'
)